In [0]:
%pip install \
    azure-keyvault-secrets==4.7.0 \
    azure-identity==1.15.0 \
    azure-core==1.29.5 \
    azure-storage-file-datalake==12.14.0 \
    sseclient-py \
    openai \
    dotenv \
    confluent-kafka \
    great-expectations \
    altair==4.2.2 \
    redis

In [0]:
dbutils.library.restartPython()

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC ## PULSE — Wikipedia EventStreams 수신 + AI 규칙 생성
# MAGIC - Wikipedia SSE → Bronze 적재
# MAGIC - AI 규칙 생성 (최초 1회)
# MAGIC - GX 품질 검사

# COMMAND ----------
import os
import sys
import json
import requests
from sseclient import SSEClient
from datetime import datetime

# ── 환경 설정 ────────────────────────────────────────────
os.environ["KEY_VAULT_URL"] = "https://kv-sense-team4.vault.azure.net/"
sys.path.insert(0, "/Workspace/Repos/3dt030@msacademy.msai.kr/3dt-3nd-project/src")

# ── vault 초기화 ─────────────────────────────────────────
import utils.vault_manager
utils.vault_manager._instance = None
from utils.vault_manager import get_vault_manager

vault = get_vault_manager()

# ── 연결 및 시크릿 로드 ──────────────────────────────────
storage_client    = vault.get_storage_client("datacopsadls")
kafka_producer    = vault.get_kafka_producer()

gx_openai_key        = vault.get_secret("gx-rulegen-openai-key")
gx_openai_endpoint   = vault.get_secret("gx-rulegen-openai-endpoint")
gx_openai_deployment = vault.get_secret("gx-rulegen-deployment-gpt-4-1-mini")
gx_openai_api_version = "2024-12-01-preview"

# ── 확인 ─────────────────────────────────────────────────
print("[OK] Key Vault 연결 완료")
print("[OK] ADLS 연결 완료")
print("[OK] Kafka Producer 연결 완료")
print("[OK] GX RuleGen OpenAI 설정 로드 완료")
print(f"[INFO] deployment = {gx_openai_deployment}")

In [0]:
import json
import requests
import time
from sseclient import SSEClient

TOPIC = "wiki-raw-events"
WIKI_STREAM_URL = "https://stream.wikimedia.org/v2/stream/recentchange"
HEADERS = {
    "Accept": "text/event-stream",
    "User-Agent": "datacops-data-quality/0.1"
}

def delivery_report(err, msg):
    if err:
        print(f"[FAIL] 전송 실패: {err}")
    else:
        print(f"[OK] offset={msg.offset()} | partition={msg.partition()}")

def collect_and_produce(stream_url, max_count=1000, filters=None, max_retries=3):
    events = []
    count = 0
    retry = 0

    while count < max_count and retry < max_retries:
        try:
            response = requests.get(stream_url, stream=True, headers=HEADERS, timeout=30)
            client = SSEClient(response)

            for event in client.events():
                if event.event != "message":
                    continue
                try:
                    data = json.loads(event.data)
                except json.JSONDecodeError:
                    continue

                if filters:
                    if not all(data.get(k) == v for k, v in filters.items()):
                        continue

                events.append(data)
                kafka_producer.produce(
                    topic=TOPIC,
                    key=str(data.get("id", "")),
                    value=json.dumps(data),
                    callback=delivery_report
                )
                kafka_producer.poll(0)
                count += 1

                if count >= max_count:
                    break

        except Exception as e:
            retry += 1
            print(f"[WARN] 연결 끊김 ({retry}/{max_retries}): {e}")
            time.sleep(2)
            continue

    kafka_producer.flush()
    print(f"[완료] {count}건 Kafka({TOPIC}) 전송 완료")
    return events

print("Wikipedia 이벤트 수집 시작...")
raw_events = collect_and_produce(stream_url=WIKI_STREAM_URL, max_count=1000)
print(f"[OK] raw_events {len(raw_events)}건 저장 완료")

### 카프카를 통해 가져온 데이터 확인

In [0]:
from confluent_kafka import Consumer
import json

consumer = Consumer({
    "bootstrap.servers": vault.get_secret("kafka-bootstrap-servers"),
    "security.protocol": "SASL_PLAINTEXT",
    "sasl.mechanism":    "SCRAM-SHA-256",
    "sasl.username":     vault.get_secret("kafka-username"),
    "sasl.password":     vault.get_secret("kafka-password"),
    "group.id":          "check-fresh",
    "auto.offset.reset": "earliest",
})

consumer.subscribe(["wiki-raw-events"])

count = 0
while count < 3:
    msg = consumer.poll(timeout=5.0)
    if msg is None:
        continue
    if msg.error():
        print(f"[ERROR] {msg.error()}")
        break
    data = json.loads(msg.value())
    print(f"\n{'='*50}")
    print(f"title   : {data.get('title')}")
    print(f"wiki    : {data.get('wiki')}")
    print(f"type    : {data.get('type')}")
    print(f"user    : {data.get('user')}")
    print(f"bot     : {data.get('bot')}")
    print(f"comment : {data.get('comment')}")
    print(f"length  : {data.get('length')}")
    print(f"revision: {data.get('revision')}")
    count += 1

consumer.close()
print(f"\n[완료] {count}건 확인")

### 수집 데이터 gx 처리 

In [0]:
# COMMAND ----------
import pandas as pd

def detect_column_type(series: pd.Series) -> str:
    non_null = series.dropna()
    if non_null.empty:
        return "unknown"
    if pd.api.types.is_bool_dtype(non_null):
        return "boolean"
    if pd.api.types.is_numeric_dtype(non_null):
        return "numeric"
    sample = non_null.astype(str).head(20)
    parsed = pd.to_datetime(sample, errors="coerce", utc=True)
    if parsed.notna().mean() >= 0.8:
        return "timestamp"
    try:
        unique_ratio = non_null.nunique() / len(non_null)
    except TypeError:
        return "string"
    if unique_ratio < 0.05:
        return "categorical"
    return "string"

def safe_sample_values(series: pd.Series, n=3):
    values = series.dropna().head(n).tolist()
    result = []
    for v in values:
        try:
            json.dumps(v)
            result.append(v)
        except TypeError:
            result.append(str(v))
    return result

def safe_unique_count(series: pd.Series) -> int:
    try:
        return int(series.nunique())
    except TypeError:
        return int(series.astype(str).nunique())

def compute_null_correlations(df: pd.DataFrame, profile: dict) -> dict:
    """
    NULL이 많은 컬럼과 다른 컬럼 값의 상관관계 자동 계산
    도메인 무관하게 데이터 패턴에서 비즈니스 의미 추론
    예: length.old NULL → type=new와 98% 일치 → allow
    """
    nullable_cols = [
        col for col, info in profile.items()
        if 0.05 < info["null_rate"] < 0.95 and col in df.columns
    ]
    categorical_cols = [
    col for col, info in profile.items()
    if info["dtype"] in ["categorical", "boolean"]
    and info["null_rate"] < 0.05
    and col in df.columns
    and 1 < info["unique_count"] < 20  # 단일값/고카디널리티 제외
    ]

    for null_col in nullable_cols:
        null_mask = df[null_col].isna()
        if null_mask.sum() == 0:
            continue

        correlations = {}
        for cat_col in categorical_cols:
            if cat_col == null_col:
                continue
            when_null = df.loc[null_mask, cat_col].value_counts(normalize=True)
            for val, ratio in when_null.items():
                if ratio >= 0.8:  # 80% 이상 일치하면 의미있는 상관관계
                    correlations[f"{cat_col}={val}"] = round(float(ratio), 3)

        if correlations:
            profile[null_col]["null_when"] = correlations
            print(f"  [상관관계 발견] {null_col}: {correlations}")

    return profile

def auto_profile(data: list[dict]) -> dict:
    df = pd.json_normalize(data)
    profile = {}
    for col in df.columns:
        series = df[col]
        non_null = series.dropna()
        dtype = detect_column_type(series)
        col_info = {
            "dtype": dtype,
            "null_rate": round(series.isna().mean(), 3),
            "unique_count": safe_unique_count(non_null),
            "sample": safe_sample_values(series, n=3)
        }
        if dtype == "numeric" and not non_null.empty:
            col_info.update({
                "mean": round(float(non_null.mean()), 3),
                "std":  round(float(non_null.std()), 3),
            })
        profile[col] = col_info

    # 상관관계 계산 추가
    print("[INFO] NULL 상관관계 분석 중...")
    profile = compute_null_correlations(df, profile)

    return profile

profile = auto_profile(raw_events)
print(f"\n[OK] 컬럼 {len(profile)}개 분석 완료")
for col, info in list(profile.items())[:10]:
    print(f"  {col}: {info['dtype']} | null={info['null_rate']} | unique={info['unique_count']}")

### 칼럼 최적화 & 도메인 자동 감지 

In [0]:
# COMMAND ----------
# null_rate 0.95 이상 컬럼만 제외하고 전체 전달

slim_profile = {
    col: info
    for col, info in profile.items()
    if info["null_rate"] < 0.95
}

print(f"[INFO] 전체 {len(profile)}개 → AI 전달 {len(slim_profile)}개 컬럼")
print(f"[INFO] 제외된 컬럼 ({len(profile) - len(slim_profile)}개):")
for col, info in profile.items():
    if info["null_rate"] >= 0.95:
        print(f"  - {col}: null={info['null_rate']}")

print("\n[INFO] 비즈니스 의미 NULL 발견된 컬럼:")
for col, info in slim_profile.items():
    if "null_when" in info:
        print(f"  - {col}: {info['null_when']}")

# ── 도메인 자동 감지 ──────────────────────────────────────
from openai import AzureOpenAI

_client = AzureOpenAI(
    api_key=gx_openai_key,
    azure_endpoint=gx_openai_endpoint,
    api_version=gx_openai_api_version,
)

def detect_domain(profile: dict) -> dict:
    """
    프로파일 컬럼명 + 샘플값으로 도메인 자동 감지
    규칙 생성 전에 실행해서 도메인 컨텍스트 확보
    어떤 도메인 데이터든 자동 감지 가능
    """
    compact = json.dumps(
        {
            col: {
                "dtype": info["dtype"],
                "sample": info["sample"]
            }
            for col, info in list(profile.items())[:15]
        },
        ensure_ascii=False,
        indent=2
    )

    response = _client.chat.completions.create(
        model=gx_openai_deployment,
        messages=[
            {
                "role": "system",
                "content": """You are a data domain expert.
Identify the data domain from column names, dtypes, and sample values.
Return a JSON with:
- domain_name: short snake_case name (e.g. wikipedia_recentchange, nyc_taxi_trips, health_checkup)
- domain_description: one sentence describing the data
- key_columns: list of 3-5 most important columns
- data_characteristics: list of 2-3 notable characteristics

Return ONLY valid JSON. No markdown."""
            },
            {
                "role": "user",
                "content": f"Column profile:\n{compact}\n\nIdentify the domain."
            }
        ],
        temperature=0,
        max_tokens=200,
    )

    raw = response.choices[0].message.content.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {
            "domain_name": "unknown_domain",
            "domain_description": "Unknown domain",
            "key_columns": [],
            "data_characteristics": []
        }

# 실행
domain_info = detect_domain(slim_profile)
domain_name = domain_info["domain_name"]

print(f"\n[OK] 도메인 감지 완료")
print(f"  domain     : {domain_name}")
print(f"  description: {domain_info['domain_description']}")
print(f"  key_columns: {domain_info['key_columns']}")
print(f"  특성       : {domain_info['data_characteristics']}")

### AI 규칙 생성 프롬프트 작성

In [0]:
# COMMAND ----------

def build_stage1_prompt(profile: dict, domain_name: str) -> list[dict]:
    """
    1단계: 전체 컬럼 → NULL 전략 + 텍스트 품질 컬럼 식별
    null_when 필드로 도메인 무관하게 비즈니스 의미 NULL 자동 판단
    """
    compact_profile = json.dumps(profile, ensure_ascii=False, indent=2)

    system_prompt = """
You are a senior data quality engineer building a domain-agnostic automated data quality platform.
The platform works for ANY domain by inferring context from column names, dtypes, null rates, and sample values.

== NULL HANDLING STRATEGY ==
- drop             : critical identifier (name contains 'id', 'uuid', 'key', 'request_id') — drop row if null
- allow            : null has business meaning
- fill_default     : fill with fixed value (specify default_value)
- fill_mean        : fill with mean (normally distributed numeric)
- fill_median      : fill with median (skewed numeric)
- fill_mode        : fill with most frequent value (categorical/boolean)
- fill_forward     : fill with previous row value (time-ordered)
- fill_backward    : fill with next row value
- fill_interpolate : linear interpolation (ordered numeric)
- fill_conditional : fill based on another column value

== NULL CORRELATION RULE (HIGHEST PRIORITY) ==
If a column has "null_when" field:
- This means NULL is strongly correlated with a specific value in another column
- This NULL has BUSINESS MEANING → ALWAYS use "allow", never fill
- Example: null_when: {"type=new": 0.98} means NULL when type=new → new page creation → allow
- This rule overrides ALL other rules

== OTHER NULL RULES ==
- Identifier columns (name contains 'id', 'uuid', 'key', 'request_id'): ALWAYS "drop"
- Boolean columns (minor, patrolled, bot): "fill_mode"
- null_rate > 0.8 columns (log_*, subtypes): "allow"
- Free-text columns (comment, title, description, parsedcomment): "fill_default" with ""
- Timestamp columns: "fill_forward" or "drop", NEVER "fill_mean" or "fill_median"
- Categorical columns with null_rate > 0.1: "fill_mode"
- Skewed numeric (high std relative to mean): "fill_median"
- Normal numeric: "fill_mean"

Return ONLY valid JSON. No markdown, no explanation, no comments.
"""

    user_prompt = f"""
Domain: {domain_name}

Column profile (dtype, null_rate, unique_count, sample, mean/std for numeric, null_when for correlated nulls):
{compact_profile}

Return:
{{
  "null_strategies": {{
    "<col>": {{
      "strategy": "drop|allow|fill_default|fill_mean|fill_median|fill_mode|fill_forward|fill_backward|fill_interpolate|fill_conditional",
      "default_value": null,
      "condition_column": null,
      "condition_map": null,
      "reason": "..."
    }}
  }},
  "text_quality_columns": [
    {{
      "column": "...",
      "checks": ["profanity", "spam", "hate_speech", "pii"],
      "reason": "..."
    }}
  ]
}}
"""
    return [
        {"role": "system", "content": system_prompt.strip()},
        {"role": "user", "content": user_prompt.strip()}
    ]


def build_stage2_prompt(profile: dict, domain_name: str) -> list[dict]:
    """
    2단계: 핵심 컬럼만 → 검증 규칙 + 이상치 탐지
    null_rate < 0.1 또는 categorical/boolean/timestamp 위주
    """
    key_profile = {
        col: info for col, info in profile.items()
        if info["null_rate"] < 0.1
        or info["dtype"] in ["categorical", "boolean", "timestamp"]
    }
    compact_profile = json.dumps(key_profile, ensure_ascii=False, indent=2)

    system_prompt = """
You are a senior data quality engineer.
Generate stable validation rules and anomaly detection rules.

== VALIDATION RULES ==
- Do NOT generate min/max range rules (sample-dependent)
- Do NOT generate string length rules (sample-dependent)
- DO generate: not null checks, type checks, allowed value sets (cardinality < 10), datetime format
- severity: critical / warning / info
- One rule per expectation type per column, no duplicates
- Categorical with high unique_count (>10) or evolving over time: WARNING not CRITICAL
- For value_set rules on columns with unique_count > 3: use WARNING not CRITICAL
- If a column has "null_when" field in profile: do NOT generate expect_column_values_to_not_be_null for it

== ANOMALY RULES ==
- delta    : size/length change columns → flag extreme changes
- zscore   : normal numeric (use mean/std from profile) → flag beyond 3 std
- iqr      : skewed numeric → flag outside 1.5*IQR
- frequency: user/bot activity → flag abnormal rates
- Always specify numeric threshold where possible

Return ONLY valid JSON. No markdown, no explanation, no comments.
"""

    user_prompt = f"""
Domain: {domain_name}
Profile:
{compact_profile}

Return:
{{
  "suite_name": "{domain_name}_quality_suite",
  "domain": "{domain_name}",
  "generated_at": "2024-01-01T00:00:00Z",
  "expectations": [
    {{
      "expectation_type": "...",
      "column": "...",
      "kwargs": {{}},
      "severity": "critical|warning|info",
      "reason": "..."
    }}
  ],
  "anomaly_rules": [
    {{
      "name": "...",
      "columns": ["..."],
      "method": "zscore|iqr|delta|frequency",
      "threshold": null,
      "severity": "critical|warning",
      "reason": "..."
    }}
  ]
}}
"""
    return [
        {"role": "system", "content": system_prompt.strip()},
        {"role": "user", "content": user_prompt.strip()}
    ]

# Redis 키 도메인 기반으로 자동 설정
CACHE_KEY_RULES  = f"gx_rules:{domain_name}"
CACHE_KEY_SCHEMA = f"gx_schema:{domain_name}"

# 프롬프트 생성
messages_stage1 = build_stage1_prompt(slim_profile, domain_name)
messages_stage2 = build_stage2_prompt(slim_profile, domain_name)
print(f"[OK] 1단계 프롬프트 준비 완료 (도메인: {domain_name})")
print(f"[OK] 2단계 프롬프트 준비 완료 (도메인: {domain_name})")
print(f"[INFO] 캐시 키: {CACHE_KEY_RULES}")

In [0]:
# COMMAND ----------
import redis
from redis.cluster import RedisCluster, ClusterNode
import hashlib
import json
from datetime import datetime

# ── Redis 연결 ────────────────────────────────────────────
redis_host = vault.get_secret("redis-host")
redis_password = vault.get_secret("redis-password")
redis_port = int(vault.get_secret("redis-port"))

# Azure Managed Redis endpoint 확인용
print(f"[INFO] Redis endpoint: {redis_host}:{redis_port}")

# Azure Managed Redis는 클러스터 모드 사용
# 주의:
# - Azure Managed Redis가 클러스터 노드 정보를 내부 IP로 반환할 수 있음
# - 이때 SSL 인증서는 도메인 기준인데, redis-py가 내부 IP로 검증하려고 해서
#   IP address mismatch 오류가 발생할 수 있음
r = RedisCluster(
    startup_nodes=[
        ClusterNode(redis_host, redis_port)
    ],
    password=redis_password,
    ssl=True,
    ssl_check_hostname=False,
    decode_responses=True,
    skip_full_coverage_check=True,
    socket_connect_timeout=10,
    socket_timeout=10,
)

r.ping()
print(f"[OK] Redis 연결 완료: {redis_host}:{redis_port}")


# ── 스키마 드리프트 감지 ──────────────────────────────────
def compute_schema_hash(profile: dict) -> str:
    schema_sig = {
        col: info["dtype"]
        for col, info in sorted(profile.items())
    }

    return hashlib.md5(
        json.dumps(schema_sig, sort_keys=True).encode()
    ).hexdigest()


def detect_schema_drift(profile: dict) -> dict:
    current_schema = {
        col: info["dtype"]
        for col, info in profile.items()
    }

    current_hash = compute_schema_hash(profile)

    stored_hash = r.get(CACHE_KEY_SCHEMA + ":hash")
    stored_schema = r.get(CACHE_KEY_SCHEMA + ":detail")

    # Redis에 기존 스키마 정보가 없으면 최초 실행으로 판단
    if not stored_hash or not stored_schema:
        return {
            "drifted": False,
            "is_first_run": True,
            "current_hash": current_hash,
            "changes": []
        }

    # 해시가 같으면 스키마 변경 없음
    if stored_hash == current_hash:
        return {
            "drifted": False,
            "is_first_run": False,
            "current_hash": current_hash,
            "changes": []
        }

    # 기존 스키마 상세 정보 로드
    prev_schema = json.loads(stored_schema)

    changes = []

    added = set(current_schema) - set(prev_schema)
    removed = set(prev_schema) - set(current_schema)

    for col in added:
        changes.append({
            "type": "added",
            "column": col,
            "dtype": current_schema[col]
        })

    for col in removed:
        changes.append({
            "type": "removed",
            "column": col,
            "dtype": prev_schema[col]
        })

    for col in set(current_schema) & set(prev_schema):
        if current_schema[col] != prev_schema[col]:
            changes.append({
                "type": "type_changed",
                "column": col,
                "from": prev_schema[col],
                "to": current_schema[col]
            })

    return {
        "drifted": True,
        "is_first_run": False,
        "current_hash": current_hash,
        "changes": changes
    }


# ── 실행 ─────────────────────────────────────────────────
drift_result = detect_schema_drift(slim_profile)

if drift_result["is_first_run"]:
    print("[INFO] 최초 실행 → 규칙 생성 필요")
    need_regenerate = True

elif drift_result["drifted"]:
    # type_changed만 있고 추가/삭제가 없으면 경고만 출력하고 규칙 재생성은 하지 않음
    real_changes = [
        c for c in drift_result["changes"]
        if c["type"] in ["added", "removed"]
    ]

    type_changes = [
        c for c in drift_result["changes"]
        if c["type"] == "type_changed"
    ]

    if type_changes and not real_changes:
        print(
            f"[INFO] 타입 변동 {len(type_changes)}개 감지 "
            "(샘플 크기 차이로 인한 변동 가능성, 규칙 유지)"
        )

        for c in type_changes:
            print(f"  ~ {c['column']}: {c['from']} → {c['to']}")

        need_regenerate = False

    else:
        print(
            f"[WARN] 스키마 드리프트 감지! "
            f"변경사항 {len(drift_result['changes'])}개:"
        )

        for c in drift_result["changes"]:
            if c["type"] == "added":
                print(f"  + 컬럼 추가: {c['column']} ({c['dtype']})")

            elif c["type"] == "removed":
                print(f"  - 컬럼 삭제: {c['column']}")

            elif c["type"] == "type_changed":
                print(
                    f"  ~ 타입 변경: {c['column']} "
                    f"({c['from']} → {c['to']})"
                )

        r.delete(CACHE_KEY_RULES)
        print("[OK] Redis 캐시 무효화 완료")

        need_regenerate = True

else:
    print("[INFO] 스키마 변경 없음 → 캐시 사용")
    need_regenerate = False

### AI 규칙 생성 

In [0]:
# COMMAND ----------
import json
from openai import AzureOpenAI

client = AzureOpenAI(
    api_key=gx_openai_key,
    azure_endpoint=gx_openai_endpoint,
    api_version=gx_openai_api_version,
)

INPUT_PRICE  = 0.40 / 1_000_000
OUTPUT_PRICE = 1.60 / 1_000_000

def call_ai(messages, label, max_tokens=8000):
    print(f"[INFO] {label} 생성 중...")
    response = client.chat.completions.create(
        model=gx_openai_deployment,
        messages=messages,
        temperature=0,
        max_tokens=max_tokens,
    )
    usage = response.usage
    cost  = (usage.prompt_tokens * INPUT_PRICE) + (usage.completion_tokens * OUTPUT_PRICE)
    print(f"  tokens: {usage.total_tokens} | cost: ${cost:.6f}")
    raw = response.choices[0].message.content.strip()
    try:
        return json.loads(raw), cost
    except json.JSONDecodeError as e:
        print(f"[WARN] JSON 파싱 실패: {e}")
        print(f"[RAW] {raw[:300]}")
        return {}, 0

# ── 캐시 히트 확인 ───────────────────────────────────────
cached = r.get(CACHE_KEY_RULES)

if cached and not need_regenerate:
    gx_rules = json.loads(cached)
    print("[OK] Redis 캐시에서 규칙 로드 (AI 호출 없음)")
    print(f"  NULL 전략  : {len(gx_rules.get('null_strategies', {}))}개")
    print(f"  검증 규칙  : {len(gx_rules.get('expectations', []))}개")
    print(f"  이상치 탐지: {len(gx_rules.get('anomaly_rules', []))}개")
    print(f"  텍스트 품질: {len(gx_rules.get('text_quality_columns', []))}개")

else:
    # ── AI 호출 ───────────────────────────────────────────
    stage1, cost1 = call_ai(messages_stage1, "1단계 (NULL 전략 + 텍스트 품질)")
    stage2, cost2 = call_ai(messages_stage2, "2단계 (검증 규칙 + 이상치 탐지)")

    gx_rules = {
        "suite_name"          : stage2.get("suite_name", ""),
        "domain"              : stage2.get("domain", ""),
        "generated_at"        : datetime.utcnow().isoformat() + "Z",
        "null_strategies"     : stage1.get("null_strategies", {}),
        "expectations"        : stage2.get("expectations", []),
        "anomaly_rules"       : stage2.get("anomaly_rules", []),
        "text_quality_columns": stage1.get("text_quality_columns", []),
    }

    total_cost = cost1 + cost2
    print(f"\n[OK] 전체 규칙 생성 완료 | 총 비용: ${total_cost:.6f}")
    print(f"  NULL 전략  : {len(gx_rules['null_strategies'])}개")
    print(f"  검증 규칙  : {len(gx_rules['expectations'])}개")
    print(f"  이상치 탐지: {len(gx_rules['anomaly_rules'])}개")
    print(f"  텍스트 품질: {len(gx_rules['text_quality_columns'])}개")

    # ── Redis에 저장 (30일 TTL) ───────────────────────────
    TTL = 60 * 60 * 24 * 30  # 30일
    current_schema = {col: info["dtype"] for col, info in slim_profile.items()}

    r.set(CACHE_KEY_RULES,                json.dumps(gx_rules, ensure_ascii=False), ex=TTL)
    r.set(CACHE_KEY_SCHEMA + ":hash",     drift_result["current_hash"],             ex=TTL)
    r.set(CACHE_KEY_SCHEMA + ":detail",   json.dumps(current_schema),               ex=TTL)

    print("[OK] Redis에 규칙 + 스키마 저장 완료 (TTL: 30일)")
    print(f"  캐시 키: {CACHE_KEY_RULES}")

In [0]:
def apply_null_strategies(df: pd.DataFrame, null_strategies: dict) -> pd.DataFrame:
    df = df.copy()
    drop_cols = []

    for col, strategy in null_strategies.items():
        if col not in df.columns:
            continue
        s = strategy["strategy"]

        if s == "drop":
            drop_cols.append(col)
        elif s == "allow":
            pass
        elif s == "fill_default":
            df[col] = df[col].fillna(strategy.get("default_value", ""))
        elif s == "fill_mean":
            df[col] = df[col].fillna(df[col].mean())
        elif s == "fill_median":
            df[col] = df[col].fillna(df[col].median())
        elif s == "fill_mode":
            mode = df[col].mode()
            if not mode.empty:
                df[col] = df[col].fillna(mode[0])
        elif s == "fill_forward":
            df[col] = df[col].ffill()
        elif s == "fill_backward":
            df[col] = df[col].bfill()
        elif s == "fill_interpolate":
            df[col] = df[col].interpolate(method="linear")
        elif s == "fill_conditional":
            cond_col = strategy.get("condition_column")
            cond_map = strategy.get("condition_map", {})
            if cond_col and cond_col in df.columns:
                for cond_val, fill_val in cond_map.items():
                    mask = df[col].isna() & (df[cond_col] == cond_val)
                    df.loc[mask, col] = fill_val
        print(f"  [{s}] {col}")

    if drop_cols:
        before = len(df)
        df = df.dropna(subset=drop_cols)
        print(f"[drop] {before - len(df)}행 제거 ({drop_cols})")

    return df

# 실행
# apply_null_strategies 실행
df = pd.json_normalize(raw_events)
df_clean = apply_null_strategies(df, gx_rules["null_strategies"])
print(f"\n[완료] 정제 전: {len(df)}행 → 정제 후: {len(df_clean)}행")

# 정제 후 NULL 현황 확인
print("\n[NULL 처리 결과]")
for col, strategy in gx_rules["null_strategies"].items():
    if col not in df_clean.columns:
        continue
    s = strategy["strategy"]
    null_after = df_clean[col].isna().sum()
    null_before = df[col].isna().sum() if col in df.columns else 0

    if s == "allow":
        print(f"  [allow] {col}: NULL {null_after}건 유지 (정상)")
    elif s == "drop":
        dropped = len(df) - len(df_clean)
        print(f"  [drop]  {col}: {dropped}행 제거")
    elif null_before > 0 and null_after == 0:
        print(f"  [{s}] {col}: NULL {null_before}건 → 0건 처리 완료")
    elif null_before > 0 and null_after > 0:
        print(f"  [{s}] {col}: NULL {null_before}건 → {null_after}건 (미처리 확인 필요)")

In [0]:
# COMMAND ----------
# 생성된 규칙 확인

import json
from collections import defaultdict

# 1. null_strategies 확인
print("=" * 50)
print("NULL 처리 전략")
print("=" * 50)
for col, strategy in gx_rules.get("null_strategies", {}).items():
    s = strategy["strategy"]
    reason = strategy.get("reason", "")
    default = strategy.get("default_value")
    cond = strategy.get("condition_column")

    if default is not None:
        print(f"  [{s}] {col} → default={default} | {reason}")
    elif cond:
        print(f"  [{s}] {col} → condition={cond} | {reason}")
    else:
        print(f"  [{s}] {col} | {reason}")

# 2. expectations severity별 분류
print("\n")
by_severity = defaultdict(list)
for exp in gx_rules.get("expectations", []):
    by_severity[exp.get("severity")].append(exp)

for severity in ["critical", "warning", "info"]:
    exps = by_severity.get(severity, [])
    if not exps:
        continue
    print("=" * 50)
    print(f"{severity.upper()} ({len(exps)}개)")
    print("=" * 50)
    for exp in exps:
        print(f"  - [{exp.get('column')}] {exp.get('expectation_type')}")
        print(f"    이유: {exp.get('reason')}")

# 3. 요약
total_strategies = len(gx_rules.get("null_strategies", {}))
total_rules      = len(gx_rules.get("expectations", []))
print(f"\n[요약] NULL 전략 {total_strategies}개 | 검증 규칙 {total_rules}개")

In [0]:
# COMMAND ----------
import pandas as pd
import numpy as np

df = df_clean
print(f"[OK] DataFrame: {df.shape}")

# ── 1. 구조 검증 ──────────────────────────────────────────
def validate_dataframe(df: pd.DataFrame, expectations: list, slim_profile: dict) -> list:
    """
    도메인 무관 검증 엔진
    - null_when 있는 컬럼 NOT NULL 규칙 자동 패스
    - value_set 실패 시 새 값 자동 확장
    """
    results = []

    for exp in expectations:
        col      = exp.get("column")
        exp_type = exp.get("expectation_type")
        kwargs   = exp.get("kwargs", {})
        severity = exp.get("severity", "warning")

        if col not in df.columns:
            continue

        series     = df[col]
        passed     = True
        fail_count = 0

        try:
            if exp_type == "expect_column_values_to_not_be_null":
                # null_when 있는 컬럼은 NULL이 정상 → 패스
                col_profile = slim_profile.get(col, {})
                if "null_when" in col_profile:
                    passed = True
                else:
                    fail_count = series.isna().sum()
                    passed     = fail_count == 0

            elif exp_type == "expect_column_values_to_be_in_set":
                allowed = kwargs.get("value_set", [])
                if allowed:
                    actual_values = series.dropna().unique().tolist()
                    new_values    = [v for v in actual_values if v not in allowed]
                    if new_values:
                        print(f"  [INFO] {col} 새 값 발견: {new_values} → 허용값 자동 확장")
                        allowed = allowed + new_values
                    fail_count = (~series.dropna().isin(allowed)).sum()
                    passed     = fail_count == 0

            elif exp_type == "expect_column_values_to_be_between":
                min_val = kwargs.get("min_value")
                max_val = kwargs.get("max_value")
                if min_val is not None and max_val is not None:
                    fail_count = (~series.between(min_val, max_val)).sum()
                    passed     = fail_count == 0

            elif exp_type == "expect_column_value_lengths_to_be_between":
                min_len = kwargs.get("min_value", 0)
                max_len = kwargs.get("max_value", 99999)
                lengths = series.dropna().astype(str).str.len()
                fail_count = (~lengths.between(min_len, max_len)).sum()
                passed     = fail_count == 0

            elif exp_type in [
                "expect_column_values_to_be_of_type",
                "expect_column_values_to_be_in_type_list",
                "expect_column_values_to_be_dateutil_parseable",
                "expect_column_values_to_be_valid_datetime",
                "expect_column_values_to_match_strftime_format",
            ]:
                passed = True

        except Exception:
            passed = True

        results.append({
            "column":     col,
            "rule":       exp_type,
            "severity":   severity,
            "passed":     passed,
            "fail_count": fail_count,
        })

    return results

# ── 2. 이상치 탐지 ────────────────────────────────────────
def apply_anomaly_rules(df: pd.DataFrame, anomaly_rules: list) -> list:
    """
    anomaly_rules 기반 이상치 탐지
    zscore / iqr / delta / frequency 지원
    """
    anomaly_results = []

    for rule in anomaly_rules:
        name     = rule.get("name")
        columns  = rule.get("columns", [])
        method   = rule.get("method")
        threshold= rule.get("threshold")
        severity = rule.get("severity", "warning")

        flagged_rows = pd.Series([False] * len(df), index=df.index)

        try:
            if method == "zscore":
                col = columns[0]
                if col in df.columns:
                    series    = df[col].dropna()
                    mean, std = series.mean(), series.std()
                    if std > 0:
                        z = (df[col] - mean) / std
                        t = threshold if threshold else 3
                        flagged_rows = z.abs() > t

            elif method == "iqr":
                col = columns[0]
                if col in df.columns:
                    series = df[col].dropna()
                    q1, q3 = series.quantile(0.25), series.quantile(0.75)
                    iqr    = q3 - q1
                    t      = threshold if threshold else 1.5
                    flagged_rows = (df[col] < q1 - t * iqr) | (df[col] > q3 + t * iqr)

            elif method == "delta":
                if len(columns) >= 2:
                    col_new, col_old = columns[0], columns[1]
                    if col_new in df.columns and col_old in df.columns:
                        delta        = df[col_new] - df[col_old]
                        t            = threshold if threshold else 10000
                        flagged_rows = delta.abs() > t

            elif method == "frequency":
                col = columns[0]
                if col in df.columns:
                    freq  = df[col].value_counts()
                    mean  = freq.mean()
                    std   = freq.std()
                    t     = threshold if threshold else 3
                    high_freq = freq[freq > mean + t * std].index
                    flagged_rows = df[col].isin(high_freq)

        except Exception as e:
            print(f"  [WARN] {name} 탐지 실패: {e}")

        flagged_count = flagged_rows.sum()
        anomaly_results.append({
            "name":          name,
            "method":        method,
            "columns":       columns,
            "severity":      severity,
            "flagged_count": int(flagged_count),
            "passed":        flagged_count == 0,
        })

    return anomaly_results

# ── 3. 실행 ───────────────────────────────────────────────
results         = validate_dataframe(df, gx_rules["expectations"], slim_profile)
anomaly_results = apply_anomaly_rules(df, gx_rules.get("anomaly_rules", []))

# ── 4. 결과 요약 ──────────────────────────────────────────
total        = len(results)
passed_count = sum(1 for r in results if r["passed"])
failed_count = total - passed_count

print(f"\n[구조 검증] 전체: {total} | 통과: {passed_count} | 실패: {failed_count}")
print(f"[품질점수] {round(passed_count/total*100, 1)}%")

if failed_count > 0:
    print("\n[실패 규칙]")
    for r in results:
        if not r["passed"]:
            print(f"  [{r['severity'].upper()}] {r['column']} | {r['rule']} | 실패건수: {r['fail_count']}")

# 이상치 탐지 결과
print(f"\n[이상치 탐지] 전체: {len(anomaly_results)}개 규칙")
for r in anomaly_results:
    status = "✅ 정상" if r["passed"] else f"⚠️ {r['flagged_count']}건 감지"
    print(f"  [{r['severity'].upper()}] {r['name']} ({r['method']}) → {status}")

#### redis 규칙 삭제 

In [0]:
# COMMAND ----------
from redis.cluster import RedisCluster, ClusterNode

redis_host = vault.get_secret("redis-host")
redis_password = vault.get_secret("redis-password")
redis_port = int(vault.get_secret("redis-port"))

redis_client = RedisCluster(
    startup_nodes=[
        ClusterNode(redis_host, redis_port)
    ],
    password=redis_password,
    ssl=True,
    ssl_check_hostname=False,
    decode_responses=True,
    skip_full_coverage_check=True,
    socket_connect_timeout=10,
    socket_timeout=10,
)

redis_client.ping()
print(f"[OK] Redis 재연결 완료: {redis_host}:{redis_port}")

# COMMAND ----------
deleted_rules = redis_client.delete(CACHE_KEY_RULES)
deleted_schema_hash = redis_client.delete(CACHE_KEY_SCHEMA + ":hash")
deleted_schema_detail = redis_client.delete(CACHE_KEY_SCHEMA + ":detail")

print("[OK] Redis 캐시 삭제 완료")
print(f"  rules 삭제 여부        : {deleted_rules}")
print(f"  schema hash 삭제 여부  : {deleted_schema_hash}")
print(f"  schema detail 삭제 여부: {deleted_schema_detail}")


# COMMAND ----------
print("rules:", redis_client.get(CACHE_KEY_RULES))
print("schema hash:", redis_client.get(CACHE_KEY_SCHEMA + ":hash"))
print("schema detail:", redis_client.get(CACHE_KEY_SCHEMA + ":detail"))

### 이상치 인덱스 수집 + 데이터 분리

In [0]:
# ── 이상치 행 인덱스 수집 ─────────────────────────────────
def collect_anomaly_indices(df: pd.DataFrame, anomaly_rules: list) -> dict:
    """
    각 anomaly rule별 flagged row index 반환
    """
    flagged = {}  # rule_name → set of indices

    for rule in anomaly_rules:
        name     = rule.get("name")
        columns  = rule.get("columns", [])
        method   = rule.get("method")
        threshold= rule.get("threshold")

        flagged_rows = pd.Series([False] * len(df), index=df.index)

        try:
            if method == "zscore":
                col = columns[0]
                if col in df.columns:
                    series = df[col].dropna()
                    mean, std = series.mean(), series.std()
                    if std > 0:
                        z = (df[col] - mean) / std
                        t = threshold if threshold else 3
                        flagged_rows = z.abs() > t

            elif method == "iqr":
                col = columns[0]
                if col in df.columns:
                    series = df[col].dropna()
                    q1, q3 = series.quantile(0.25), series.quantile(0.75)
                    iqr = q3 - q1
                    t = threshold if threshold else 1.5
                    flagged_rows = (df[col] < q1 - t * iqr) | (df[col] > q3 + t * iqr)

            elif method == "delta":
                if len(columns) >= 2:
                    col_new, col_old = columns[0], columns[1]
                    if col_new in df.columns and col_old in df.columns:
                        delta = df[col_new] - df[col_old]
                        t = threshold if threshold else 10000
                        flagged_rows = delta.abs() > t

            elif method == "frequency":
                col = columns[0]
                if col in df.columns:
                    freq = df[col].value_counts()
                    mean = freq.mean()
                    std  = freq.std()
                    t    = threshold if threshold else 3
                    high_freq = freq[freq > mean + t * std].index
                    flagged_rows = df[col].isin(high_freq)

        except Exception as e:
            print(f"  [WARN] {name} 인덱스 수집 실패: {e}")

        flagged[name] = set(df[flagged_rows].index.tolist())

    return flagged

# 실행
anomaly_index_map = collect_anomaly_indices(df_clean, gx_rules.get("anomaly_rules", []))

# 전체 이상치 인덱스 합집합
all_anomaly_idx = set()
for rule_name, idx_set in anomaly_index_map.items():
    if idx_set:
        print(f"  [{rule_name}] → {len(idx_set)}건")
        all_anomaly_idx.update(idx_set)

print(f"\n[분리] 총 이상치 행: {len(all_anomaly_idx)}건")

# ── Silver / Quarantine 분리 ──────────────────────────────
df_silver     = df_clean.drop(index=list(all_anomaly_idx)).reset_index(drop=True)
df_quarantine = df_clean.loc[list(all_anomaly_idx)].copy()

print(f"[Silver]     정상 데이터: {len(df_silver)}행")
print(f"[Quarantine] 격리 데이터: {len(df_quarantine)}행")

### 격리 이유 태깅

In [0]:
# ── 격리 이유 태깅 ────────────────────────────────────────
processed_at = datetime.utcnow().isoformat() + "Z"
run_ts       = datetime.utcnow().strftime("%H%M%S")  

def tag_quarantine_reasons(df_q: pd.DataFrame, anomaly_index_map: dict, anomaly_rules: list) -> pd.DataFrame:
    df_q = df_q.copy()

    rule_meta     = {r["name"]: r for r in anomaly_rules}
    reason_list   = {idx: [] for idx in df_q.index}
    severity_list = {idx: [] for idx in df_q.index}

    for rule_name, idx_set in anomaly_index_map.items():
        meta = rule_meta.get(rule_name, {})
        sev  = meta.get("severity", "warning").upper()
        for idx in idx_set:
            if idx in reason_list:
                reason_list[idx].append(rule_name)
                severity_list[idx].append(sev)

    # 격리 사유 컬럼
    df_q["_quarantine_reason"]   = df_q.index.map(lambda i: ", ".join(reason_list.get(i, ["UNKNOWN"])))
    df_q["_quarantine_severity"] = df_q.index.map(lambda i: severity_list.get(i, ["WARNING"])[0])
    df_q["_quarantine_ts"]       = processed_at
    # 추적 컬럼
    df_q["_processed_at"]        = processed_at
    df_q["_run_id"]              = run_ts

    return df_q

df_quarantine = tag_quarantine_reasons(df_quarantine, anomaly_index_map, gx_rules.get("anomaly_rules", []))
print("[OK] 격리 이유 태깅 완료")
print(df_quarantine[["_quarantine_reason", "_quarantine_severity", "_quarantine_ts"]].value_counts().to_string())

### Silver 메타 컬럼 추가

In [0]:
# ── Silver: 추적 컬럼만 추가 ──────────────────────────────
df_silver["_processed_at"] = processed_at
df_silver["_run_id"]       = run_ts

print(f"[OK] Silver 메타 추가 완료: {df_silver.shape}")

### ADLS 저장 (Silver + Quarantine)

In [0]:
import io
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.types import NullType, StringType
from pyspark.sql.functions import lit

def fix_void_columns(df_spark):
    """VOID(NullType) 컬럼 → 드롭"""
    void_cols = [f.name for f in df_spark.schema.fields
                 if isinstance(f.dataType, NullType)]
    if void_cols:
        print(f"  [DROP] 전체 NULL 컬럼 제거: {void_cols}")
        df_spark = df_spark.drop(*void_cols)
    return df_spark

def save_by_event_type(df_pandas, base_path: str, type_col: str = "type"):
    if type_col in df_pandas.columns:
        for event_type, subset in df_pandas.groupby(type_col):
            null_rate = subset.isnull().mean()
            drop_cols = null_rate[null_rate >= 0.99].index.tolist()
            subset    = subset.drop(columns=drop_cols).copy()

            path     = f"{base_path}/event_type={event_type}/run={run_ts}"
            df_spark = pandas_to_spark_safe(subset, spark)  # ← 교체
            df_spark = fix_void_columns(df_spark)
            df_spark.write.mode("overwrite").parquet(path)
            print(f"  [OK] {event_type}: {len(subset)}행 x {len(subset.columns)}열 → {path}")
    else:
        null_rate = df_pandas.isnull().mean()
        drop_cols = null_rate[null_rate >= 0.99].index.tolist()
        df_pandas = df_pandas.drop(columns=drop_cols).copy()

        df_spark = pandas_to_spark_safe(df_pandas, spark)   # ← 교체
        df_spark = fix_void_columns(df_spark)
        df_spark.write.mode("overwrite").parquet(f"{base_path}/run={run_ts}")
        print(f"  [OK] {len(df_pandas)}행 x {len(df_pandas.columns)}열 → {base_path}/run={run_ts}")

def pandas_to_spark_safe(df_pandas, spark):
    """pandas → Spark 안전 변환 — 모든 컬럼 타입을 명시적으로 지정"""
    from pyspark.sql.types import (
        StructType, StructField,
        StringType, LongType, DoubleType, BooleanType, NullType
    )
    import numpy as np

    df = df_pandas.copy()

    # 1단계: 모든 컬럼을 안전한 Python 타입으로 변환
    for col in df.columns:
        series = df[col].dropna()
        if len(series) == 0:
            continue

        # numpy 타입 → Python 기본 타입으로 변환
        has_bool  = any(isinstance(v, (bool, np.bool_)) for v in series)
        has_float = any(isinstance(v, (float, np.floating)) for v in series)
        has_int   = any(isinstance(v, (int, np.integer)) for v in series)
        has_complex = any(isinstance(v, (list, dict)) for v in series)

        # bool + 숫자 혼합 or 복잡한 타입 → String
        if (has_bool and (has_float or has_int)) or has_complex:
            print(f"  [FIX] {col} → String")
            df[col] = df[col].apply(
                lambda x: str(x) if not (isinstance(x, float) and pd.isna(x)) and x is not None else None
            )
        elif has_bool:
            # bool만 있으면 명시적으로 bool 변환
            df[col] = df[col].apply(
                lambda x: bool(x) if pd.notna(x) else None
            )

    # 2단계: 스키마 자동 생성
    fields = []
    for col in df.columns:
        series = df[col].dropna()
        if len(series) == 0:
            fields.append(StructField(col, StringType(), True))
            continue

        sample = series.iloc[0]
        if isinstance(sample, bool):
            fields.append(StructField(col, BooleanType(), True))
        elif isinstance(sample, (int, np.integer)):
            fields.append(StructField(col, LongType(), True))
        elif isinstance(sample, (float, np.floating)):
            fields.append(StructField(col, DoubleType(), True))
        else:
            fields.append(StructField(col, StringType(), True))

    schema = StructType(fields)

    # 3단계: Python 기본 타입으로 변환 후 Spark DataFrame 생성
    records = []
    for _, row in df.iterrows():
        record = {}
        for col in df.columns:
            v = row[col]
            if pd.isna(v) if not isinstance(v, (list, dict, str)) else False:
                record[col] = None
            elif isinstance(v, np.bool_):
                record[col] = bool(v)
            elif isinstance(v, np.integer):
                record[col] = int(v)
            elif isinstance(v, np.floating):
                record[col] = float(v)
            else:
                record[col] = v
        records.append(record)

    return spark.createDataFrame(records, schema=schema)

In [0]:
spark   = SparkSession.getActiveSession()
today   = datetime.utcnow().strftime("%Y-%m-%d")

CONTAINER = "silver"
ACCOUNT   = "datacopsadls"
BASE_PATH = f"abfss://{CONTAINER}@{ACCOUNT}.dfs.core.windows.net"



# ── Silver 저장 ───────────────────────────────────────────
print("[Silver 저장]")
silver_base = f"{BASE_PATH}/wikipedia_events/date={today}"
save_by_event_type(df_silver, silver_base, type_col="type")

# ── Quarantine 저장 ───────────────────────────────────────
print("\n[Quarantine 저장]")
for reason in df_quarantine["_quarantine_reason"].unique():
    subset      = df_quarantine[df_quarantine["_quarantine_reason"] == reason]
    safe_reason = reason.replace(", ", "_").replace(" ", "_")
    q_base      = f"{BASE_PATH}/quarantine/source=wikipedia/date={today}/reason={safe_reason}"
    save_by_event_type(subset, q_base, type_col="type")

print(f"\n[완료] Silver {len(df_silver)}행 / Quarantine {len(df_quarantine)}행 저장")

### GX 실행 로그 저장

In [0]:
# ── GX 실행 로그 저장 ─────────────────────────────────────
gx_log = {
    "run_id":          processed_at,
    "source":          "wikipedia_events",
    "date":            today,
    "total_rules":     total,
    "passed":          passed_count,
    "failed":          failed_count,
    "quality_score":   round(passed_count / total * 100, 1),
    "silver_rows":     len(df_silver),
    "quarantine_rows": len(df_quarantine),
    "anomaly_summary": [
        {
            "name":          r["name"],
            "method":        r["method"],
            "flagged_count": r["flagged_count"],
            "passed":        r["passed"],
            "severity":      r["severity"],
        }
        for r in anomaly_results
    ],
}

# Spark로 JSON 저장 (storage_client 사용 안 함)
log_path = f"{BASE_PATH}/logs/gx_runs/date={today}/run_{processed_at.replace(':', '-')}.json"
log_json = json.dumps(gx_log, ensure_ascii=False, indent=2)

log_df = spark.createDataFrame([(log_json,)], ["log"])
log_df.write.mode("overwrite").text(log_path)

print(f"[OK] GX 로그 저장: {log_path}")
print(log_json)